# Energy-Aware Workload Scheduler — Demo

Schedules 3 compute jobs to minimize energy cost and carbon emissions.

| Job | Duration | Deadline |
|-----|----------|----------|
| ETL pipeline | 2 hr | 6 hours from now |
| Model training | 6 hr | Tomorrow 9:00 AM |
| Data export | 1 hr | 3 hours from now |

In [ ]:
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

from scheduler.energy_data import generate_synthetic_data, get_hourly_forecast
from scheduler.engine import schedule_job

In [ ]:
# Fixed anchor so demo output is reproducible regardless of when you run it
NOW = datetime(2026, 5, 17, 8, 0)

df = generate_synthetic_data(days=7, start_date=datetime(2026, 5, 17, 0, 0))
print(f"Generated {len(df)} hourly records: {df.timestamp.min()} → {df.timestamp.max()}")
df.head()

In [ ]:
jobs = [
    {"name": "ETL pipeline",   "duration_hours": 2, "deadline": NOW + timedelta(hours=6)},
    {"name": "Model training", "duration_hours": 6, "deadline": datetime(2026, 5, 18, 9, 0)},
    {"name": "Data export",    "duration_hours": 1, "deadline": NOW + timedelta(hours=3)},
]

results = []
for job in jobs:
    forecast = get_hourly_forecast(NOW, job["deadline"], df=df)
    result = schedule_job(job["duration_hours"], job["deadline"], forecast, now=NOW)
    results.append(result)

    print(f"\n{'='*58}")
    print(f"  Job:               {job['name']} ({job['duration_hours']} hr)")
    print(f"  Deadline:          {job['deadline'].strftime('%b %d %H:%M')}")
    print(f"  Recommended start: {result['recommended_start'].strftime('%b %d %H:%M')}")
    print(f"  Avg price:         ${result['avg_price']:.2f}/MWh")
    print(f"  Avg carbon:        {result['avg_carbon']:.0f} g CO\u2082/kWh")
    print(f"  Cost savings:      {result['savings_vs_naive_pct']:.1f}% vs starting now")
    print(f"  Carbon savings:    {result['carbon_savings_vs_naive_pct']:.1f}% vs starting now")
    print(f"  Naive baseline:    start {result['naive_baseline']['start'].strftime('%b %d %H:%M')}  "
          f"${result['naive_baseline']['avg_price']:.2f}/MWh")
    print(f"  Top 3 windows:")
    for i, w in enumerate(result["candidate_windows"]):
        tag = "  <-- recommended" if i == 0 else ""
        print(f"    {i+1}. {w['start'].strftime('%b %d %H:%M')}\u2013{w['end'].strftime('%H:%M')}"
              f"  ${w['avg_price']:.2f}/MWh  {w['avg_carbon']:.0f} g CO\u2082/kWh{tag}")

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))

# Show price and carbon from midnight today through 10am tomorrow
plot_df = df[df["timestamp"] < pd.Timestamp(datetime(2026, 5, 18, 10, 0))].copy()

ax1.plot(plot_df["timestamp"], plot_df["price_per_mwh"],
         color="#1f77b4", linewidth=2, label="Energy price ($/MWh)")
ax1.set_ylabel("Price ($/MWh)", color="#1f77b4")
ax1.tick_params(axis="y", labelcolor="#1f77b4")

ax2 = ax1.twinx()
ax2.plot(plot_df["timestamp"], plot_df["carbon_intensity"],
         color="#d62728", linestyle="--", linewidth=1.5, alpha=0.5,
         label="Carbon intensity (g CO\u2082/kWh)")
ax2.set_ylabel("Carbon intensity (g CO\u2082/kWh)", color="#d62728")
ax2.tick_params(axis="y", labelcolor="#d62728")

colors = ["#2ca02c", "#ff7f0e", "#9467bd"]
for job, result, color in zip(jobs, results, colors):
    start = result["recommended_start"]
    end   = result["recommended_end"]
    label = (f"{job['name']}: {start.strftime('%H:%M')}\u2013{end.strftime('%H:%M')}"
             f"  ({result['savings_vs_naive_pct']:.0f}% cheaper)")
    ax1.axvspan(start, end, alpha=0.3, color=color, label=label)

ax1.axvline(NOW, color="black", linestyle=":", linewidth=2, label="Now (08:00)")

ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %d\n%H:%M"))
ax1.xaxis.set_major_locator(mdates.HourLocator(interval=3))

handles, labels = ax1.get_legend_handles_labels()
ax1.legend(handles, labels, loc="upper right", fontsize=8.5)
ax1.set_title("Recommended Job Schedule — Energy Price & Carbon Forecast", fontsize=13)
ax1.set_xlabel("Time")
fig.tight_layout()
plt.savefig("schedule_demo.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved schedule_demo.png")